In [1]:
import subprocess
import sys
from pathlib import Path

folder = Path.cwd()
env = folder / '.q2_clean_python'
runner = env / 'bin' / 'python'
if not runner.exists():
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'uv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '--seed', str(env)], check=True)
    packages = ['torch==2.6.0', 'transformers==4.51.3', 'peft==0.15.2', 'datasets==3.6.0', 'accelerate==1.7.0', 'bitsandbytes==0.50.2', 'python-docx>=1.1', 'matplotlib', 'pandas']
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(runner), *packages], check=True)


In [2]:
from pathlib import Path

source = 'import os\nimport re\nimport gc\nimport csv\nimport json\nimport ast\nimport math\nimport time\nimport random\nimport hashlib\nfrom pathlib import Path\nfrom collections import Counter\n\nimport torch\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom datasets import load_dataset\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup\nfrom peft import LoraConfig, AdaLoraConfig, get_peft_model, prepare_model_for_kbit_training\nfrom docx import Document\nfrom docx.shared import Pt, Inches\n\nSEED = 42\nMODEL = \'Qwen/Qwen2.5-1.5B-Instruct\'\nDATASET = \'glaiveai/glaive-function-calling-v2\'\nROOT = Path(\'Q2_Quick_Output\')\nROOT.mkdir(exist_ok=True)\nEPOCHS = 1\nMAX_LENGTH = 192\nMAX_NEW_TOKENS = 24\nLR = 2e-4\nTRAIN_BATCH = 1\nGRAD_ACCUM = 1\nEVAL_BATCH = 4\nDATASET_SIZE = 2000\nFAST_TRAIN_EXAMPLES = 8\nFAST_EVAL_EXAMPLES = 25\nTARGET_MODULES = [\'q_proj\', \'k_proj\', \'v_proj\', \'o_proj\']\n\n\ndef seed_everything():\n    random.seed(SEED)\n    torch.manual_seed(SEED)\n    torch.cuda.manual_seed_all(SEED)\n    torch.backends.cudnn.deterministic = False\n    torch.backends.cudnn.benchmark = True\n    torch.backends.cuda.matmul.allow_tf32 = True\n\n\ndef native_tool_names(system):\n    return sorted(set(re.findall(r\'"name"\\s*:\\s*"([^"\\\\]+)"\', system)))\n\n\ndef normalize_arguments(args):\n    if isinstance(args, str):\n        try:\n            args = json.loads(args)\n        except (ValueError, TypeError):\n            try:\n                args = ast.literal_eval(args)\n            except (ValueError, SyntaxError, TypeError):\n                return None\n    return args if isinstance(args, dict) else None\n\n\ndef parse_first_exchange(item):\n    system = str(item.get(\'system\') or \'\')\n    chat = str(item.get(\'chat\') or \'\')\n    match = re.search(r\'ASSISTANT:\\s*\', chat)\n    if match is None:\n        return None\n    prev = chat[:match.start()]\n    users = list(re.finditer(r\'USER:\\s*\', prev))\n    if not users:\n        return None\n    user = prev[users[-1].end():].strip()\n    response = chat[match.end():].split(\'<|endoftext|>\', 1)[0].strip()\n    if not user or not response:\n        return None\n    allowed = native_tool_names(system)\n    if \'<functioncall>\' in response:\n        content = response.split(\'<functioncall>\', 1)[1].strip()\n        try:\n            obj = json.loads(content)\n        except (ValueError, TypeError):\n            try:\n                obj = ast.literal_eval(content)\n            except (ValueError, SyntaxError, TypeError, MemoryError):\n                return None\n        if not isinstance(obj, dict):\n            return None\n        name = obj.get(\'name\')\n        arguments = normalize_arguments(obj.get(\'arguments\', {}))\n        if not isinstance(name, str) or arguments is None or name not in allowed:\n            return None\n        label = {\'name\': name, \'arguments\': arguments}\n        kind = \'tool\'\n    else:\n        label = {\'name\': \'none\', \'arguments\': {}}\n        kind = \'no_tool\'\n    return {\'system\': system, \'user\': user, \'label\': label, \'kind\': kind, \'allowed\': allowed}\n\n\ndef get_examples():\n    file = ROOT / \'split_2000.json\'\n    old_split = Path(\'Q2_Fast_Output/split_2000.json\')\n    if not file.exists() and old_split.exists():\n        file.write_text(old_split.read_text(encoding=\'utf-8\'), encoding=\'utf-8\')\n    if file.exists():\n        split = json.loads(file.read_text(encoding=\'utf-8\'))\n        if len(split.get(\'train\', [])) == 1800 and len(split.get(\'test\', [])) == 200:\n            return split\n    pool = []\n    seen = set()\n    for item in load_dataset(DATASET, split=\'train\', streaming=True):\n        parsed = parse_first_exchange(item)\n        if parsed is None:\n            continue\n        digest = hashlib.sha256((parsed[\'system\'] + \'\\n\' + parsed[\'user\']).encode(\'utf-8\')).hexdigest()\n        if digest in seen:\n            continue\n        seen.add(digest)\n        parsed[\'id\'] = digest[:16]\n        pool.append(parsed)\n        if len(pool) == DATASET_SIZE:\n            break\n    if len(pool) != DATASET_SIZE:\n        raise RuntimeError(\'The dataset did not provide 2,000 valid unique examples\')\n    random.Random(SEED).shuffle(pool)\n    split = {\'train\': pool[:1800], \'test\': pool[1800:]}\n    file.write_text(json.dumps(split, ensure_ascii=False), encoding=\'utf-8\')\n    return split\n\ndef make_prompt(tokenizer, row):\n    system = \'Return exactly one JSON object with keys name and arguments. Use only functions explicitly available below. If no function is appropriate, return {"name":"none","arguments":{}}. Do not add explanations.\\n\' + row[\'system\']\n    msgs = [{\'role\': \'system\', \'content\': system}, {\'role\': \'user\', \'content\': row[\'user\']}]\n    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n\n\ndef encode_train(tokenizer, row):\n    prompt = tokenizer(make_prompt(tokenizer, row), add_special_tokens=False)[\'input_ids\']\n    answer = json.dumps(row[\'label\'], ensure_ascii=False, separators=(\',\', \':\'), sort_keys=True)\n    answer_ids = tokenizer(answer + tokenizer.eos_token, add_special_tokens=False)[\'input_ids\']\n    if len(answer_ids) >= MAX_LENGTH:\n        raise RuntimeError(\'Target exceeds maximum token length\')\n    prompt = prompt[-(MAX_LENGTH - len(answer_ids)):]\n    ids = prompt + answer_ids\n    labels = [-100] * len(prompt) + answer_ids\n    return ids, labels\n\n\ndef load_model(quantized, dtype):\n    checkpoint = torch.cuda.get_device_properties(0).total_memory < 35 * 2**30\n    if quantized:\n        qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)\n        base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=qc, device_map={\'\': 0}, attn_implementation=\'sdpa\')\n        return prepare_model_for_kbit_training(base, use_gradient_checkpointing=checkpoint)\n    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype, attn_implementation=\'sdpa\', low_cpu_mem_usage=True).cuda()\n    if checkpoint:\n        base.gradient_checkpointing_enable()\n        base.enable_input_require_grads()\n    return base\n\n\ndef configuration(method, rank, total_steps):\n    common = {\'target_modules\': TARGET_MODULES, \'lora_alpha\': 16, \'lora_dropout\': 0.0, \'bias\': \'none\', \'task_type\': \'CAUSAL_LM\'}\n    if method == \'adalora\':\n        return AdaLoraConfig(init_r=12 if rank == 8 else max(rank + 1, round(rank * 1.5)), target_r=rank, total_step=total_steps, tinit=max(1, round(total_steps * .10)), tfinal=max(1, round(total_steps * .10)), deltaT=max(1, round(total_steps * .02)), orth_reg_weight=0.1, **common)\n    return LoraConfig(r=rank, use_dora=(method == \'dora\'), **common)\n\n\ndef delete_model(model):\n    del model\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\ndef extract_output(text):\n    try:\n        val = json.loads(text.strip())\n    except (ValueError, TypeError):\n        return None\n    if not isinstance(val, dict) or set(val) != {\'name\', \'arguments\'}:\n        return None\n    if not isinstance(val[\'name\'], str):\n        return None\n    args = normalize_arguments(val[\'arguments\'])\n    if args is None:\n        return None\n    return {\'name\': val[\'name\'], \'arguments\': args}\n\n\ndef evaluate(model, tokenizer, test, label, rank, prepared=None):\n    model.eval()\n    model.config.use_cache = True\n    if prepared is None:\n        prepared = prepare_test(tokenizer, test)\n    rows = []\n    new_tokens = 0\n    torch.cuda.synchronize()\n    t_start = time.perf_counter()\n    with torch.inference_mode():\n        for start in range(0, len(prepared), EVAL_BATCH):\n            sample = prepared[start:start+EVAL_BATCH]\n            seq_len = max(len(x[1]) for x in sample)\n            ids = torch.full((len(sample), seq_len), tokenizer.pad_token_id, dtype=torch.long, device=\'cuda\')\n            mask = torch.zeros_like(ids)\n            for ix, (_, sequence) in enumerate(sample):\n                ids[ix, -len(sequence):] = torch.tensor(sequence, dtype=torch.long, device=\'cuda\')\n                mask[ix, -len(sequence):] = 1\n            result = model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1, use_cache=True, pad_token_id=tokenizer.eos_token_id)\n            answers = result[:, ids.shape[1]:]\n            texts = tokenizer.batch_decode(answers, skip_special_tokens=True)\n            for (item, _), tokens, raw in zip(sample, answers, texts):\n                matches = (tokens == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]\n                new_tokens += int(matches[0].item()+1) if len(matches) else tokens.numel()\n                raw = raw.strip()\n                got = extract_output(raw)\n                truth = item[\'label\']\n                valid = got is not None\n                tool_match = valid and got[\'name\'] == truth[\'name\']\n                args_match = tool_match and got[\'arguments\'] == truth[\'arguments\']\n                unwanted = valid and (got[\'name\'] not in item[\'allowed\'] and got[\'name\'] != \'none\' or got[\'name\'] != \'none\' and truth[\'name\'] == \'none\')\n                unauthorized_refund = valid and \'refund\' in got[\'name\'].lower() and got[\'name\'] != truth[\'name\']\n                if not valid:\n                    error = \'broken JSON\'\n                elif unwanted:\n                    error = \'unauthorized or unnecessary tool\'\n                elif not tool_match:\n                    error = \'wrong tool\'\n                elif not args_match:\n                    error = \'wrong or made-up arguments\'\n                else:\n                    error = \'\'\n                rows.append({\'id\': item[\'id\'], \'message\': item[\'user\'], \'reference\': json.dumps(truth, ensure_ascii=False, sort_keys=True), \'prediction\': raw, \'json_valid\': int(valid), \'tool_correct\': int(tool_match), \'full_match\': int(args_match), \'unnecessary\': int(unwanted), \'unsafe_refund\': int(unauthorized_refund), \'error\': error})\n    torch.cuda.synchronize()\n    duration = time.perf_counter() - t_start\n    with (ROOT / f\'predictions_{label}_r{rank}.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as file:\n        writer = csv.DictWriter(file, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    return {\'json_valid\': sum(r[\'json_valid\'] for r in rows), \'tool_correct\': sum(r[\'tool_correct\'] for r in rows), \'full_match\': sum(r[\'full_match\'] for r in rows), \'unnecessary\': sum(r[\'unnecessary\'] for r in rows), \'unsafe_refund\': sum(r[\'unsafe_refund\'] for r in rows), \'inference_seconds\': duration, \'output_tps\': new_tokens / duration}\n\n\ndef prepare_test(tokenizer, test):\n    prepared = []\n    for row in test:\n        prompt = make_prompt(tokenizer, row)\n        sequence = tokenizer(prompt, add_special_tokens=False)[\'input_ids\'][-(MAX_LENGTH-MAX_NEW_TOKENS):]\n        prepared.append((row, sequence))\n    prepared.sort(key=lambda row: len(row[1]))\n    return prepared\n\n\ndef train_one(model, tokenizer, train, method, rank, dtype, encoded=None):\n    if encoded is None:\n        encoded = [encode_train(tokenizer, row) for row in train]\n    effective_batch = TRAIN_BATCH * GRAD_ACCUM\n    num_steps = math.ceil(len(encoded) / effective_batch) * EPOCHS\n    model = get_peft_model(model, configuration(method, rank, num_steps))\n    model.config.use_cache = False\n    model.train()\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    trainable_parameters = [p for p in model.parameters() if p.requires_grad]\n    optimizer = torch.optim.AdamW(trainable_parameters, lr=LR)\n    schedule = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=max(1, round(num_steps*.03)), num_training_steps=num_steps)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=dtype == torch.float16)\n    gc.collect()\n    torch.cuda.empty_cache()\n    torch.cuda.reset_peak_memory_stats()\n    torch.cuda.synchronize()\n    started = time.perf_counter()\n    step = 0\n    optimizer.zero_grad(set_to_none=True)\n    for epoch in range(EPOCHS):\n        order = list(range(len(encoded)))\n        random.Random(SEED + epoch).shuffle(order)\n        grouped = []\n        for first in range(0, len(order), 128):\n            grouped.extend(sorted(order[first:first+128], key=lambda idx: len(encoded[idx][0])))\n        for group_start in range(0, len(grouped), effective_batch):\n            group = grouped[group_start:group_start+effective_batch]\n            microbatches = [group[j:j+TRAIN_BATCH] for j in range(0, len(group), TRAIN_BATCH)]\n            for group_ix in microbatches:\n                part = [encoded[idx] for idx in group_ix]\n                length = math.ceil(max(len(pair[0]) for pair in part)/8)*8\n                ids = torch.full((len(part), length), tokenizer.pad_token_id, dtype=torch.long, device=\'cuda\')\n                attention = torch.zeros_like(ids)\n                labels = torch.full_like(ids, -100)\n                for j, (sequence, answer) in enumerate(part):\n                    count = len(sequence)\n                    ids[j,:count] = torch.tensor(sequence, dtype=torch.long, device=\'cuda\')\n                    attention[j,:count] = 1\n                    labels[j,:count] = torch.tensor(answer, dtype=torch.long, device=\'cuda\')\n                with torch.autocast(\'cuda\', dtype=dtype):\n                    loss = model(input_ids=ids, attention_mask=attention, labels=labels).loss\n                scaler.scale(loss / len(microbatches)).backward()\n                del loss, ids, attention, labels\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(trainable_parameters, 1.0)\n            scaler.step(optimizer)\n            scaler.update()\n            step += 1\n            if method == \'adalora\':\n                model.base_model.update_and_allocate(step)\n            schedule.step()\n            optimizer.zero_grad(set_to_none=True)\n    torch.cuda.synchronize()\n    minutes = (time.perf_counter() - started) / 60\n    memory_gib = torch.cuda.max_memory_allocated() / 2**30\n    del optimizer, schedule, scaler\n    gc.collect()\n    torch.cuda.empty_cache()\n    return model, trainable, minutes, memory_gib\n\n\ndef run_configuration(method, rank, split, tokenizer, dtype, base_params, encoded=None, prepared=None):\n    seed_everything()\n    model = load_model(method == \'qlora\', dtype)\n    model, trainable, minutes, memory = train_one(model, tokenizer, split[\'train\'], method, rank, dtype, encoded=encoded)\n    scores = evaluate(model, tokenizer, split[\'test\'], method, rank, prepared=prepared)\n    name = method + \'_r\' + str(rank)\n    model.save_pretrained(ROOT / \'adapters\' / name)\n    row = {\'method\': method, \'rank\': rank, \'trainable_params\': trainable, \'trainable_pct\': trainable / (base_params + trainable) * 100, \'peak_training_gib\': memory, \'training_minutes\': minutes, **scores}\n    delete_model(model)\n    return row\n\n\ndef save_metrics(metrics):\n    (ROOT / \'metrics.json\').write_text(json.dumps(metrics, indent=2), encoding=\'utf-8\')\n    with (ROOT / \'results.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(metrics[0]))\n        writer.writeheader()\n        writer.writerows(metrics)\n\n\ndef gather():\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'A CUDA GPU is required for the training experiments\')\n    started = time.perf_counter()\n    seed_everything()\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    tokenizer = AutoTokenizer.from_pretrained(MODEL)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    split = get_examples()\n    if len(split[\'train\']) != 1800 or len(split[\'test\']) != 200:\n        raise RuntimeError(\'Incorrect split sizes\')\n    encoded = []\n    for item in split[\'train\']:\n        try:\n            sample = encode_train(tokenizer, item)\n        except RuntimeError:\n            continue\n        encoded.append(sample)\n        if len(encoded) == FAST_TRAIN_EXAMPLES:\n            break\n    if len(encoded) < FAST_TRAIN_EXAMPLES:\n        raise RuntimeError(\'Not enough training samples for the quick test\')\n    quick_test = split[\'test\'][:FAST_EVAL_EXAMPLES]\n    prepared = prepare_test(tokenizer, quick_test)\n    metrics_path = ROOT / \'metrics.json\'\n    metrics = json.loads(metrics_path.read_text(encoding=\'utf-8\')) if metrics_path.exists() else []\n    names = {(row[\'method\'], row[\'rank\']) for row in metrics}\n    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype, attn_implementation=\'sdpa\').cuda()\n    base_params = sum(p.numel() for p in base.parameters())\n    if (\'base\', 0) not in names:\n        torch.cuda.reset_peak_memory_stats()\n        scores = evaluate(base, tokenizer, quick_test, \'base\', 0, prepared=prepared)\n        metrics.append({\'method\': \'base\', \'rank\': 0, \'trainable_params\': 0, \'trainable_pct\': 0, \'peak_training_gib\': None, \'training_minutes\': 0, **scores})\n        save_metrics(metrics)\n    delete_model(base)\n    for method in (\'lora\', \'qlora\', \'dora\', \'adalora\'):\n        if (method, 8) not in {(r[\'method\'], r[\'rank\']) for r in metrics}:\n            metrics.append(run_configuration(method, 8, {\'train\':split[\'train\'],\'test\':quick_test}, tokenizer, dtype, base_params, encoded=encoded, prepared=prepared))\n            save_metrics(metrics)\n    core = [r for r in metrics if r[\'method\'] != \'base\' and r[\'rank\'] == 8]\n    winner = sorted(core, key=lambda row: (-row[\'full_match\'], -row[\'tool_correct\'], row[\'peak_training_gib\'], row[\'training_minutes\']))[0][\'method\']\n    for rank in (4, 16, 32):\n        if (winner, rank) not in {(r[\'method\'], r[\'rank\']) for r in metrics}:\n            metrics.append(run_configuration(winner, rank, {\'train\':split[\'train\'],\'test\':quick_test}, tokenizer, dtype, base_params, encoded=encoded, prepared=prepared))\n            save_metrics(metrics)\n    ranks = sorted([r for r in metrics if r[\'method\'] == winner], key=lambda row: row[\'rank\'])\n    plt.figure(figsize=(6.5, 4))\n    plt.plot([r[\'rank\'] for r in ranks], [100*r[\'tool_correct\'] / FAST_EVAL_EXAMPLES for r in ranks], marker=\'o\', color=\'black\')\n    plt.xlabel(\'Rank\')\n    plt.ylabel(\'Tool accuracy (%)\')\n    plt.xticks([4, 8, 16, 32])\n    plt.tight_layout()\n    plt.savefig(ROOT / \'rank_vs_accuracy.png\', dpi=180)\n    plt.close()\n    make_report(metrics, winner)\n    (ROOT / \'runtime.json\').write_text(json.dumps({\'elapsed_minutes\':(time.perf_counter()-started)/60,\'gpu\':torch.cuda.get_device_name(0),\'train_batch\':TRAIN_BATCH,\'gradient_accumulation\':GRAD_ACCUM,\'effective_train_batch\':TRAIN_BATCH*GRAD_ACCUM,\'eval_batch\':EVAL_BATCH,\'max_length\':MAX_LENGTH,\'max_new_tokens\':MAX_NEW_TOKENS,\'epochs_on_fast_subset\':EPOCHS,\'fast_training_examples_per_run\':FAST_TRAIN_EXAMPLES,\'train_pool_size\':len(split[\'train\']),\'test_pool_size\':len(split[\'test\']),\'test_evaluated\':FAST_EVAL_EXAMPLES,\'quick_diagnostic\':True},indent=2),encoding=\'utf-8\')\n    return metrics\n\n\ndef fmt_score(value):\n    return f\'{100*value / FAST_EVAL_EXAMPLES:.1f}%\'\n\n\ndef make_report(metrics, winner):\n    doc = Document()\n    normal = doc.styles[\'Normal\']\n    normal.font.name = \'Arial\'\n    normal.font.size = Pt(10.5)\n    normal.paragraph_format.space_after = Pt(4)\n    section = doc.sections[0]\n    section.top_margin = Inches(.65)\n    section.bottom_margin = Inches(.65)\n    section.left_margin = Inches(.8)\n    section.right_margin = Inches(.8)\n    doc.add_paragraph(\'Question 2 - Results\')\n    doc.add_paragraph(\'Quick diagnostic experiment on Qwen2.5-1.5B-Instruct. Data: 2,000 real Glaive examples (1,800 train, 200 held out, but only 25 evaluated in this fast mode). For speed, only 8 training examples were used for one epoch for each adapter. The same 8 examples and training settings were used in all seven training runs. Every model was evaluated on the same 25 of the 200 held-out examples; generation was limited to 24 tokens. This is not a full 1,800-example training experiment. QLoRA used NF4.\')\n    doc.add_paragraph(\'Run / trainable parameters / trainable % / peak training GiB / training minutes / valid JSON / tool correct / full match / unnecessary calls / output tokens per second:\')\n    priority = {\'base\': 0, \'lora\': 1, \'qlora\': 2, \'dora\': 3, \'adalora\': 4}\n    for r in sorted(metrics, key=lambda r: (0 if r[\'rank\'] in (0,8) else 1, priority[r[\'method\']], r[\'rank\'])):\n        mem = \'N/A\' if r[\'peak_training_gib\'] is None else f"{r[\'peak_training_gib\']:.2f}"\n        doc.add_paragraph(f"{r[\'method\']} rank {r[\'rank\']}: {r[\'trainable_params\']:,} / {r[\'trainable_pct\']:.4f}% / {mem} / {r[\'training_minutes\']:.2f} / {r[\'json_valid\']}/{FAST_EVAL_EXAMPLES} / {r[\'tool_correct\']}/{FAST_EVAL_EXAMPLES} / {r[\'full_match\']}/{FAST_EVAL_EXAMPLES} / {r[\'unnecessary\']} / {r[\'output_tps\']:.2f}")\n    chosen = next(r for r in metrics if r[\'method\'] == winner and r[\'rank\'] == 8)\n    doc.add_paragraph(f"For this quick diagnostic only, {winner} rank 8 gave {chosen[\'full_match\']}/{FAST_EVAL_EXAMPLES} exact tool-and-argument matches and {chosen[\'tool_correct\']}/{FAST_EVAL_EXAMPLES} correct tools, with {chosen[\'peak_training_gib\']:.2f} GiB peak training memory and {chosen[\'training_minutes\']:.2f} minutes training. Its measured output speed here was {chosen[\'output_tps\']:.2f} tokens/s. These short-run results are not enough to justify production deployment.")\n    doc.add_paragraph(\'Rank experiment: \' + \', \'.join(f"rank {r[\'rank\']} = {fmt_score(r[\'tool_correct\'])}" for r in sorted([r for r in metrics if r[\'method\'] == winner], key=lambda row: row[\'rank\'])))\n    with (ROOT / f\'predictions_{winner}_r8.csv\').open(newline=\'\', encoding=\'utf-8\') as stream:\n        mistakes = [r for r in csv.DictReader(stream) if r[\'error\']][:5]\n    doc.add_paragraph(\'Five actual mistakes from the winning model:\')\n    if not mistakes:\n        doc.add_paragraph(\'No incorrect examples were found in this quick test subset.\')\n    for i, r in enumerate(mistakes, 1):\n        doc.add_paragraph(f"{i}. Message: {r[\'message\'][:250]}\\nCorrect: {r[\'reference\'][:300]}\\nGot: {r[\'prediction\'][:300]}\\nError: {r[\'error\']}")\n    if mistakes:\n        errors = Counter(row[\'error\'] for row in mistakes)\n        doc.add_paragraph(\'Among these examples, the recurring mistakes were: \' + \', \'.join(f\'{k} ({v})\' for k,v in errors.items()) + \'.\')\n    doc.add_paragraph(\'Using separate small LoRA adapters for router, refund, and shipping agents keeps one frozen base model in memory instead of storing three complete model copies. A real live risk is routing one request through the wrong adapter or changing adapters while another request is still using that model.\')\n    all_main = [r for r in metrics if r[\'rank\'] == 8 and r[\'method\'] != \'base\']\n    memory_min = min(all_main, key=lambda r:r[\'peak_training_gib\'])\n    accuracy_best = max(all_main, key=lambda r:r[\'full_match\'])\n    doc.add_paragraph(f"What stood out in this actual run: {memory_min[\'method\']} used the least peak training VRAM ({memory_min[\'peak_training_gib\']:.2f} GiB), while {accuracy_best[\'method\']} got the most exact matches ({accuracy_best[\'full_match\']}/{FAST_EVAL_EXAMPLES}). The rank curve shows whether more adapter parameters actually helped here.")\n    doc.save(ROOT / \'Q2_Measured_Report.docx\')\n\n\nif __name__ == \'__main__\':\n    gather()\n'
Path('Q2_Experiments_Quick.py').write_text(source, encoding='utf-8')


23096

In [3]:
import os
import sys
import subprocess
from pathlib import Path

configuration = os.environ.copy()
configuration['MPLBACKEND'] = 'Agg'
configuration['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

with Path('Q2_Quick_Run_Log.txt').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(
        [str(runner), '-u', 'Q2_Experiments_Quick.py'],
        env=configuration,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        log.write(line)
        log.flush()
    code = process.wait()

if code:
    raise RuntimeError('Check Q2_Quick_Run_Log.txt for the full error')


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
/content/.q2_clean_python/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/content/.q2_clean_python/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/content/.q2_clean_python/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `

In [4]:
import pandas as pd
from pathlib import Path
from IPython.display import display, FileLink

output = Path('Q2_Quick_Output')
metrics = pd.read_csv(output / 'results.csv')
display(metrics)
for item in ['Q2_Measured_Report.docx', 'results.csv', 'rank_vs_accuracy.png', 'runtime.json']:
    file = output / item
    if file.exists():
        display(FileLink(str(file)))


,method,rank,trainable_params,trainable_pct,peak_training_gib,training_minutes,json_valid,tool_correct,full_match,unnecessary,unsafe_refund,inference_seconds,output_tps
0,base,0,0,0.000000,NaN,0.000000,5,2,2,3,0,21.685958,26.007612
1,lora,8,2179072,0.140959,6.138031,0.111703,5,2,2,3,0,21.818791,24.520148
2,qlora,8,2179072,0.140959,5.265508,0.116368,2,0,0,2,0,19.460535,28.519257
3,dora,8,2279424,0.147441,6.139092,0.122087,5,2,2,3,0,30.727513,17.411107
4,adalora,8,3269952,0.211376,6.199410,0.125203,5,2,2,3,0,23.142056,23.809466
5,lora,4,1089536,0.070529,6.125977,0.106499,5,2,2,3,0,22.917047,23.388703
6,lora,16,4358144,0.281521,6.162507,0.108523,5,2,2,3,0,22.956022,24.045978
7,lora,32,8716288,0.561461,6.211213,0.108275,5,2,2,3,0,23.149156,23.154192


/content/Q2_Quick_Output/Q2_Measured_Report.docx

/content/Q2_Quick_Output/results.csv

/content/Q2_Quick_Output/rank_vs_accuracy.png

/content/Q2_Quick_Output/runtime.json

In [5]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import display, FileLink

root = Path.cwd()
output = root / "Q2_Quick_Output"
archive = root / "Question_2_All_Results.zip"

with ZipFile(archive, "w", ZIP_DEFLATED, compresslevel=1) as z:
    for file in output.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(root))

    for name in ["Q2_Experiments_Quick.py", "Q2_Quick_Run_Log.txt"]:
        file = root / name
        if file.is_file():
            z.write(file, file.name)

try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    display(FileLink(str(archive)))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>